# 3. FX图模式量化

## 3.1. FX图模式-动态量化

### (1) 引入模块

In [1]:
import warnings
# 忽略警告
warnings.filterwarnings("ignore")
import torch
import copy
import torch.ao.quantization.quantize_fx as quantize_fx    

### (2) 加载模型

In [2]:
def load_alex_model():
    import torch
    import torchvision
    # 1. 加载模型
    model = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
    return model

In [3]:
org_model = load_alex_model()
org_model

AlexNet(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
    (1): ReLU(inplace=True)
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (4): ReLU(inplace=True)
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU(inplace=True)
    (10): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2d(output_size=(6, 6))
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=9216, out_features=4096, bias=True)
 

### (3) 加载数据集

In [4]:
# 返回DataLoader对象。
def load_data(root="F:/04Datasets/ImageNet2012", split="val"):
    """
        split只支持"train"与"val"
    """
    import torchvision.transforms as transforms
    from torchvision.datasets import ImageNet
    from torch.utils.data import DataLoader
    import torch
    import torchvision
    # 加载数据集
    ds_imagenet2012 = ImageNet(
        root=root,
        split=split,
        transform = torchvision.models.AlexNet_Weights.IMAGENET1K_V1.transforms() # 需要是对象
        # target_transform=None,   # 标签转换
        # loader=Image.open   # 默认（还可以直接加载为Tensor：）
    )
    # 取部分子集
    num_calibration = 1000   # 总样本是50000 
    num_calibration = num_calibration if num_calibration<=len(ds_imagenet2012) else len(ds_imagenet2012)
    torch.manual_seed(42)
    indices = torch.randperm(len(ds_imagenet2012))[:num_calibration] + 1  # +1是因为randperm生成0-999
    subsets_imagenet2012 = torch.utils.data.Subset(ds_imagenet2012, indices)

    loader_imagenet2012 = DataLoader(
        dataset=subsets_imagenet2012,        # 单样本数据集
        batch_size=100,   # 数据集批次大小
        shuffle=False,  # 是否随机洗牌数据集 
    )
    return loader_imagenet2012



In [5]:
inputs_loader = load_data()
for x, y in inputs_loader:
    print(x.shape, y.shape)
    break

torch.Size([100, 3, 224, 224]) torch.Size([100])


### (4) 验证原模型的推理准确率

In [6]:
org_model.eval()
num_total = 0 
num_corre = 0
for x, y in inputs_loader:
    y_ = org_model(x)

    prob, cls_id = torch.max(y_, dim=1)
    num_corre += (cls_id == y).sum().item()
    num_total += len(cls_id)

accu = num_corre * 100.0 / num_total
print(F"准确率：{accu:.2f}%")

准确率：56.20%


### (5) 动态量化

In [7]:
def dynamic_quantization_example(model, inputs):
    # 创建模型实例
    model.eval()
    # 获取原始模型的输出
    with torch.no_grad():
        original_output = model(inputs)
    
    print(f"\n原始模型输出: {original_output}")
    print(f"原始模型输出数据类型: {original_output.dtype}")
    
    # 4. 配置动态量化
    # 对于动态量化，我们通常只量化线性层和LSTM层
    # qconfig_dict = {
    #     "": torch.ao.quantization.default_dynamic_qconfig,
    # }
    qconfig_dict = torch.ao.quantization.QConfigMapping().set_global(torch.ao.quantization.default_dynamic_qconfig)
    # 或者更精确地指定哪些模块需要量化
    # qconfig_dict = {
    #     "fc1": torch.ao.quantization.default_dynamic_qconfig,
    #     "fc2": torch.ao.quantization.default_dynamic_qconfig,
    #     "fc3": torch.ao.quantization.default_dynamic_qconfig,
    # }
    
    # 5. 准备动态量化
    print(f"\n准备动态量化...")
    model_prepared = quantize_fx.prepare_fx(model, qconfig_dict, example_inputs=(inputs,))
    
    # 6. 转换为量化模型
    print(f"转换为量化模型...")
    model_quantized = quantize_fx.convert_fx(model_prepared)
    
    # 打印量化后的模型结构
    print(f"\n量化后的模型结构:")
    model_quantized.print_readable()
    
    # 7. 测试量化模型
    with torch.no_grad():
        quantized_output = model_quantized(inputs)
    
    print(f"\n量化模型输出 (前5个值): {quantized_output}")
    print(f"量化模型输出数据类型: {quantized_output.dtype}")
    
    # 8. 比较输出差异
    diff = torch.abs(original_output - quantized_output)
    print(f"\n输出差异统计:")
    print(f"  最大差异: {diff.max().item():.6f}")
    print(f"  平均差异: {diff.mean().item():.6f}")
    print(f"  中位数差异: {diff.median().item():.6f}")
    
    return model_quantized

In [8]:
x, y = list(inputs_loader)[0]
fx_qmodel = dynamic_quantization_example(load_alex_model(), x)


原始模型输出: tensor([[-6.0874, -4.5001,  1.8768,  ..., -4.3960, -1.6043,  1.3684],
        [-4.1702,  1.3669, -2.0138,  ..., -3.7139, -0.3009,  1.1762],
        [ 3.2602, -1.0565,  2.6399,  ...,  1.4871, -0.7978,  2.8241],
        ...,
        [-4.8405, -0.7147,  0.7123,  ..., -3.8969,  1.7570,  1.9236],
        [-1.6047, -1.1635, -1.4302,  ...,  2.5278, -0.6948, -0.5232],
        [-2.1237,  1.3875,  2.9244,  ..., -6.8505, -2.2117,  1.6213]])
原始模型输出数据类型: torch.float32

准备动态量化...
转换为量化模型...

量化后的模型结构:
class GraphModule(torch.nn.Module):
    def forward(self, x : torch.Tensor) -> torch.Tensor:
         # File: C:\Users\ThinkPad\AppData\Roaming\Python\Python313\site-packages\torch\fx\proxy.py:251 in create_proxy, code: user_frame_summary = CapturedTraceback.extract().summary()
        features_0 = getattr(self.features, "0")(x);  x = None
        features_2 = getattr(self.features, "2")(features_0);  features_0 = None
        features_3 = getattr(self.features, "3")(features_2);  features_2 =

In [9]:
fx_qmodel

GraphModule(
  (features): Module(
    (0): ConvReLU2d(
      (0): Conv2d(3, 64, kernel_size=(11, 11), stride=(4, 4), padding=(2, 2))
      (1): ReLU(inplace=True)
    )
    (2): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): ConvReLU2d(
      (0): Conv2d(64, 192, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
      (1): ReLU(inplace=True)
    )
    (5): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): ConvReLU2d(
      (0): Conv2d(192, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
    )
    (8): ConvReLU2d(
      (0): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
    )
    (10): ConvReLU2d(
      (0): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU(inplace=True)
    )
    (12): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (avgpool): AdaptiveAvgPool2

In [10]:
fx_qmodel.graph

### (6) 验证量化模型推理效果

In [11]:
fx_qmodel.eval()
num_total = 0 
num_corre = 0
for x, y in load_data():
    y_ = fx_qmodel(x)

    prob, cls_id = torch.max(y_, dim=1)
    num_corre += (cls_id == y).sum().item()
    num_total += len(cls_id)

accu = num_corre * 100.0 / num_total
print(F"准确率：{accu:.2f}%")

准确率：56.00%


### (7) 注意

- FX图模式动态量化的核心是指定动态配置：(两种方式都可以)
    - 重点是使用`torch.ao.quantization.default_dynamic_qconfig`配置。该配置使用参数is_dynamic=True

```python
qconfig_dict = {
    "": torch.ao.quantization.default_dynamic_qconfig,
}
qconfig_dict = torch.ao.quantization.QConfigMapping().set_global(torch.ao.quantization.default_dynamic_qconfig)
```

In [12]:
torch.ao.quantization.default_dynamic_qconfig

QConfig(activation=functools.partial(<class 'torch.ao.quantization.observer.PlaceholderObserver'>, dtype=torch.quint8, quant_min=0, quant_max=255, is_dynamic=True){}, weight=functools.partial(<class 'torch.ao.quantization.observer.MinMaxObserver'>, dtype=torch.qint8, qscheme=torch.per_tensor_symmetric){})

## 3.2. FX图模式-静态量化

- FX图模式静态量化在编程模式上与FX图模式动态量化一致，唯一的区别是在设置配置的时候，使用is_dynamic=False的配置，如下：

In [13]:
torch.ao.quantization.get_default_qconfig("fbgemm")

QConfig(activation=functools.partial(<class 'torch.ao.quantization.observer.HistogramObserver'>, reduce_range=True){}, weight=functools.partial(<class 'torch.ao.quantization.observer.PerChannelMinMaxObserver'>, dtype=torch.qint8, qscheme=torch.per_channel_symmetric){})

### (1) 引入模块

In [1]:
import warnings
# 忽略警告
warnings.filterwarnings("ignore")
import torch
import copy
import torch.ao.quantization.quantize_fx as quantize_fx

### (2) 加载模型

In [2]:
def load_vit_model():
    import torch
    import torchvision
    # 1. 加载模型
    model = torchvision.models.vit_b_16(weights=torchvision.models.ViT_B_16_Weights.DEFAULT)
    # model = torchvision.models.alexnet(weights=torchvision.models.AlexNet_Weights.DEFAULT)
    model = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
    return model

In [3]:
vit_model = load_vit_model()
# vit_model

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\ThinkPad/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|█████████████████████████████████████████████████████████████████████████████| 97.8M/97.8M [00:10<00:00, 9.50MB/s]


### (3) 加载数据集

In [4]:
# 返回DataLoader对象。
def load_imagenet_data(root="F:/04Datasets/ImageNet2012", split="val"):
    """
        split只支持"train"与"val"
    """
    import torchvision.transforms as transforms
    from torchvision.datasets import ImageNet
    from torch.utils.data import DataLoader
    import torch
    import torchvision
    # 加载数据集
    ds_imagenet2012 = ImageNet(
        root=root,
        split=split,
        # transform = torchvision.models.AlexNet_Weights.DEFAULT.transforms() # 需要是对象
        # transform = torchvision.models.ViT_B_16_Weights.DEFAULT.transforms() # 需要是对象
        transform = torchvision.models.ResNet50_Weights.DEFAULT.transforms() # 需要是对象
        # target_transform=None,   # 标签转换
        # loader=Image.open   # 默认（还可以直接加载为Tensor：）
    )
    # 取部分子集
    num_calibration = 1000   # 总样本是50000 
    num_calibration = num_calibration if num_calibration<=len(ds_imagenet2012) else len(ds_imagenet2012)
    torch.manual_seed(42)
    indices = torch.randperm(len(ds_imagenet2012))[:num_calibration] + 1  # +1是因为randperm生成0-999
    subsets_imagenet2012 = torch.utils.data.Subset(ds_imagenet2012, indices)

    loader_imagenet2012 = DataLoader(
        dataset=subsets_imagenet2012,        # 单样本数据集
        batch_size=100,   # 数据集批次大小
        shuffle=False,  # 是否随机洗牌数据集 
    )
    return loader_imagenet2012

In [5]:
imagenet_loader = load_imagenet_data()
for x, y in imagenet_loader:
    print(x.shape, y.shape)
    break

torch.Size([100, 3, 224, 224]) torch.Size([100])


### (4) 验证原模型推理准确率

In [6]:
def evel_model(model, loader):
    model.eval()
    num_total = 0 
    num_corre = 0
    for x, y in loader:
        y_ = model(x)
    
        prob, cls_id = torch.max(y_, dim=1)
        num_corre += (cls_id == y).sum().item()
        num_total += len(cls_id)
    
    accu = num_corre * 100.0 / num_total
    print(F"准确率：{accu:.2f}%")

In [7]:
evel_model(vit_model, imagenet_loader)

准确率：78.60%


### (5) 静态量化

In [8]:
def dynamic_quantization_example(model, loader):
    x, y = list(loader)[0]
    # 创建模型实例
    model.eval()
    # 获取原始模型的输出
    with torch.no_grad():
        original_output = model(x)
    
    print(f"\n原始模型输出: {original_output[0, :10]}")
    print(f"原始模型输出数据类型: {original_output.dtype}")
    
    # 1. 配置动态量化
    # 对于动态量化，我们通常只量化线性层和LSTM层
    # qconfig_dict = {
    #     "": torch.ao.quantization.default_dynamic_qconfig,
    # }
    qconfig_dict = torch.ao.quantization.QConfigMapping().set_global(torch.ao.quantization.get_default_qconfig("fbgemm"))
    
    # 2. 准备动态量化
    print(f"\n准备静态量化...")
    model_prepared = quantize_fx.prepare_fx(model, qconfig_dict, example_inputs=(x,))

    # 3. 校准模型
    print(f"\n模型校准...")
    with torch.no_grad():
        for i, (inputs, _) in enumerate(loader):
            print(F"\t校准批次-1{i:03d}")
            model_prepared(inputs)
    
    # 4. 转换为量化模型
    print(f"转换为量化模型...")
    model_quantized = quantize_fx.convert_fx(model_prepared)
    
    # 打印量化后的模型结构
    # print(f"\n量化后的模型结构:")
    # model_quantized.print_readable()
    
    # 5. 测试量化模型
    with torch.no_grad():
        quantized_output = model_quantized(x)
    
    print(f"\n量化模型输出: {quantized_output[0, :10]}")
    print(f"量化模型输出数据类型: {quantized_output.dtype}")
    
    # 8. 比较输出差异
    diff = torch.abs(original_output - quantized_output)
    print(f"\n输出差异统计:")
    print(f"  最大差异: {diff.max().item():.6f}")
    print(f"  平均差异: {diff.mean().item():.6f}")
    print(f"  中位数差异: {diff.median().item():.6f}")
    
    return model_quantized

In [9]:
fx_static_model = dynamic_quantization_example(vit_model, imagenet_loader)


原始模型输出: tensor([-0.2544, -0.0631, -0.0424, -0.3812,  0.1024, -0.1224, -0.0627, -0.3761,
        -0.0912, -0.0019])
原始模型输出数据类型: torch.float32

准备静态量化...

模型校准...
	校准批次-1000
	校准批次-1001
	校准批次-1002
	校准批次-1003
	校准批次-1004
	校准批次-1005
	校准批次-1006
	校准批次-1007
	校准批次-1008
	校准批次-1009
转换为量化模型...

量化模型输出: tensor([-0.2493, -0.1662, -0.0831, -0.2493,  0.0831, -0.0831, -0.0831, -0.3324,
         0.0000,  0.0000])
量化模型输出数据类型: torch.float32

输出差异统计:
  最大差异: 2.102870
  平均差异: 0.076739
  中位数差异: 0.054713


In [10]:
fx_static_model

GraphModule(
  (conv1): QuantizedConvReLU2d(3, 64, kernel_size=(7, 7), stride=(2, 2), scale=0.30994588136672974, zero_point=0, padding=(3, 3))
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Module(
    (0): Module(
      (conv1): QuantizedConvReLU2d(64, 64, kernel_size=(1, 1), stride=(1, 1), scale=0.1371077448129654, zero_point=0)
      (conv2): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(1, 1), scale=0.1879856437444687, zero_point=0, padding=(1, 1))
      (conv3): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.2747594118118286, zero_point=67)
      (downsample): Module(
        (0): QuantizedConv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), scale=0.4421340227127075, zero_point=80)
      )
    )
    (1): Module(
      (conv1): QuantizedConvReLU2d(256, 64, kernel_size=(1, 1), stride=(1, 1), scale=0.10525482147932053, zero_point=0)
      (conv2): QuantizedConvReLU2d(64, 64, kernel_size=(3, 3), stride=(

### (6) 验证量化模型推理准确率

In [11]:
evel_model(fx_static_model, imagenet_loader)

准确率：78.40%


### (7) 注意

- 动态量化通常保持较高精度，但加速有限
- 静态量化可能精度略降，但加速明显
- 准确率差异取决于：
    - 模型架构（CNN vs Linear）
    - 权重分布范围与激活值分布：长尾分布难以量化
    - 校准数据质量：静态量化依赖校准数据
    - 层融合效果: 静态量化可以融合操作
    - 异常值处理: 动态量化更灵活
    - 通常差异在1-3%以内，但某些模型可能更大
- 一般应对策略：
    - 混合精度量化:
        - 敏感层保留FP32
        - 其他层使用INT8
    - 优化量化参数:
        - 使用不同的observer类型
        - 调整量化范围（如使用对称/非对称量化）

- **准确率差异总结：**
    - **一般情况下差异不大**（< 1%）：
        - 当模型设计良好时
        - 当校准数据有代表性时
        - 当使用合适的量化配置时
    - **可能差异较大**（> 3%）：
        - 模型包含极端值或长尾分布
        - 静态量化校准数据不具代表性
        - 模型对数值精度敏感
        - 输入分布剧烈变化
    - **动态量化通常更稳定**：
        - 因为激活动态调整
        - 适合输入分布变化的场景
    - **静态量化需要更多调优**：
        - 但一旦调好，性能最好
        - 适合固定输入场景

- **最佳实践建议：**
    - 先用动态量化快速部署
    - 如果需要更高性能，尝试静态量化
    - 如果静态量化损失大，使用QAT
    - 最后考虑混合精度量化方案

## 3.3. FX图模式-训练感知量化

### (1) 引入模块

In [1]:
import warnings
# 忽略警告
warnings.filterwarnings("ignore")
import torch
import copy
import torch.ao.quantization.quantize_fx as quantize_fx
import torchvision
from torch.utils.data import DataLoader, Subset
import torch.ao.quantization as tq
import os

### (2) LeNet-5神经网络模型实现

In [2]:
import torch
class LeNet5(torch.nn.Module):
    def __init__(self, cls_num=10):
        super(LeNet5, self).__init__()
        self.conv1 = torch.nn.Conv2d(1, 6, 5, padding=2)
        self.relu1 = torch.nn.ReLU(inplace=True)
        self.pool1 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.conv2 = torch.nn.Conv2d(6, 16, 5)
        self.relu2 = torch.nn.ReLU(inplace=True)
        self.pool2 = torch.nn.MaxPool2d(kernel_size=2, stride=2)
        
        self.fc3   = torch.nn.Linear(16 * 5 * 5,  120)
        self.relu3 = torch.nn.ReLU(inplace=True)
        
        self.fc4   = torch.nn.Linear(120, 84)
        self.relu4 = torch.nn.ReLU(inplace=True)
        
        self.fc5   = torch.nn.Linear(84, cls_num)

    
    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = torch.flatten(x, 1)
        
        x = self.fc3(x)
        x = self.relu3(x)
        
        x = self.fc4(x)
        x = self.relu4(x)

        x = self.fc5(x)
        return x


In [3]:
model_fp32 = LeNet5()
model_fp32

LeNet5(
  (conv1): Conv2d(1, 6, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
  (relu1): ReLU(inplace=True)
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(6, 16, kernel_size=(5, 5), stride=(1, 1))
  (relu2): ReLU(inplace=True)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc3): Linear(in_features=400, out_features=120, bias=True)
  (relu3): ReLU(inplace=True)
  (fc4): Linear(in_features=120, out_features=84, bias=True)
  (relu4): ReLU(inplace=True)
  (fc5): Linear(in_features=84, out_features=10, bias=True)
)

### (3) 加载数据集

In [4]:
def get_minst_laoder(root="./data", batch_size=128, num_calibration_batches=10):
    """
        num_calibration_batches用于校准的数据集批数。
    """
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    train_dataset = torchvision.datasets.MNIST(root=root, train=True,  download=True, transform=transform)
    test_dataset  = torchvision.datasets.MNIST(root=root, train=False, download=True, transform=transform)
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader =  torch.utils.data.DataLoader(test_dataset,  batch_size=batch_size, shuffle=False)
    # 校准数据集
    indices = torch.randperm(len(train_dataset))[:num_calibration_batches * batch_size]
    calibration_dataset = torch.utils.data.Subset(train_dataset, indices)
    calibration_loader = torch.utils.data.DataLoader(calibration_dataset, batch_size=batch_size, shuffle=False) 
    return train_loader, test_loader, calibration_loader

In [5]:
train_loader, test_loader, calibration_loader = get_minst_laoder(batch_size=1000)

### (4) 模型评估

In [6]:
def validate_model(model, test_loader, device="cpu"):
    model.eval()
    model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            y_ = model(x)
            _, pred = y_.max(1)
            total += y.size(0)
            correct += (pred==y).sum().item()
    accuracy = 100. * correct / total
    return accuracy

In [7]:
validate_model(model_fp32, test_loader)

12.2

### (5) 量化感知训练与量化

In [8]:
# 量化感知训练主流程
def quantize_and_train(model, train_loader, test_loader, epoches=10, device="cuda"):
    # ============== 开始量化感知训练 ==============
    qat_model = copy.deepcopy(model)
    # 设置为训练模式
    qat_model.train()
    qat_model = qat_model.to('cpu') # 首先将模型移动到CPU，因为量化操作通常在CPU上进行
    # 配置量化器
    qconfig = tq.get_default_qat_qconfig('x86')  # 对于x86 CPU使用fbgemm
    # 创建QConfig映射
    qconfig_mapping = torch.ao.quantization.QConfigMapping().set_global(qconfig)
    
    # 准备模型进行QAT（FX图模式）
    
    # 准备QAT模型
    example_inputs = (list(test_loader)[0][0],)
    qat_model = torch.ao.quantization.quantize_fx.prepare_qat_fx(
        qat_model, 
        qconfig_mapping, 
        example_inputs
    )
    # 将模型移回设备进行训练
    qat_model.to(device)
    # 定义QAT的优化器
    qat_optimizer = torch.optim.Adam(qat_model.parameters(), lr=0.0001)  # 使用较小的学习率
    criterion = torch.nn.CrossEntropyLoss()       
    # 量化感知训练
    
    for epoch in range(epoches):  # 训练2个epoch作为示例
        train_loss = 0
        correct = 0
        total = 0
        qat_model.train()
        for i, (x, y) in enumerate(train_loader):
            x, y = x.to(device), y.to(device)
            qat_optimizer.zero_grad()
            y_ = qat_model(x)
            loss = criterion(y_, y)
            loss.backward()
            qat_optimizer.step()
            
            train_loss += loss.item()
            _, predicted = y_.max(1)
            total += y.size(0)
            correct += predicted.eq(y).sum().item()
        accuracy = 100. * correct / total
        avg_loss = train_loss / len(train_loader)
        print(F"轮数：{epoch+1:02d},损失：{avg_loss:.2f},准确率：{accuracy:.2f}%")

    # ============== 转换为量化模型 ==============
    qat_model.to("cpu")
    qat_model.eval()
    quantized_model = torch.ao.quantization.quantize_fx.convert_fx(qat_model)
    # 保存原始模型
    torch.save(quantized_model.state_dict(), "lenet5_quantized.pth")
    torch.save(model.state_dict(), "baseline_model.pth")
    # 比较两个模型大小
    baseline_size = os.path.getsize("baseline_model.pth") / 1024
    quantized_size = os.path.getsize("lenet5_quantized.pth") / 1024
    print(f"量化模型大小压缩: {baseline_size / quantized_size:.2f}倍")
    os.remove("lenet5_quantized.pth")
    os.remove("baseline_model.pth")
    
    return model, quantized_model

In [9]:
model_fp32, quantized_model = quantize_and_train(model_fp32, train_loader, test_loader, epoches=10, device="cuda")
quantized_model

轮数：01,损失：2.28,准确率：39.59%
轮数：02,损失：2.18,准确率：74.27%
轮数：03,损失：1.99,准确率：83.50%
轮数：04,损失：1.62,准确率：86.24%
轮数：05,损失：0.99,准确率：87.61%
轮数：06,损失：0.49,准确率：88.96%
轮数：07,损失：0.35,准确率：90.22%
轮数：08,损失：0.30,准确率：91.20%
轮数：09,损失：0.28,准确率：91.87%
轮数：10,损失：0.26,准确率：92.40%
量化模型大小压缩: 3.29倍


GraphModule(
  (conv1): QuantizedConvReLU2d(1, 6, kernel_size=(5, 5), stride=(1, 1), scale=0.018276281654834747, zero_point=0, padding=(2, 2))
  (pool1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): QuantizedConvReLU2d(6, 16, kernel_size=(5, 5), stride=(1, 1), scale=0.04599085822701454, zero_point=0)
  (pool2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc3): QuantizedLinearReLU(in_features=400, out_features=120, scale=0.23245573043823242, zero_point=0, qscheme=torch.per_channel_affine)
  (fc4): QuantizedLinearReLU(in_features=120, out_features=84, scale=0.13049553334712982, zero_point=0, qscheme=torch.per_channel_affine)
  (fc5): QuantizedLinear(in_features=84, out_features=10, scale=0.20956489443778992, zero_point=102, qscheme=torch.per_channel_affine)
)

### (6) 量化模型推理准确率

In [10]:
validate_model(quantized_model, test_loader)

93.12

### (7) 总结

- 在量化感知训练量化中，使用的准备函数是：
    - `torch.ao.quantization.quantize_fx.prepare_qat_fx`

----